# Exercise - Practical implementation of POS & NER

In [1]:
import nltk
import spacy
import re
import sys
import pandas as pd
import matplotlib as plt
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from IPython import display
sys.modules['IPython.core.display'] = display

### Load Data

In [2]:
bbc_news = pd.read_csv("./bbc_news.csv")
#bbc_news.head(10)

In [3]:
bbc_news.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Unnamed: 0   1000 non-null   int64 
 1   index        1000 non-null   int64 
 2   title        1000 non-null   object
 3   pubDate      1000 non-null   object
 4   guid         1000 non-null   object
 5   link         1000 non-null   object
 6   description  1000 non-null   object
dtypes: int64(2), object(5)
memory usage: 54.8+ KB


In [4]:
titles = pd.DataFrame(bbc_news['title'])
titles.head()

,title
0,Can I refuse to work?
1,'Liz Truss the Brief?' World reacts to UK poli...
2,Rationing energy is nothing new for off-grid c...
3,The hunt for superyachts of sanctioned Russian...
4,Platinum Jubilee: 70 years of the Queen in 70 ...


### Clean Data

In [5]:
# convert titles to lower case
titles['lowercase'] = titles['title'].str.lower()
titles.head()

,title,lowercase
0,Can I refuse to work?,can i refuse to work?
1,'Liz Truss the Brief?' World reacts to UK poli...,'liz truss the brief?' world reacts to uk poli...
2,Rationing energy is nothing new for off-grid c...,rationing energy is nothing new for off-grid c...
3,The hunt for superyachts of sanctioned Russian...,the hunt for superyachts of sanctioned russian...
4,Platinum Jubilee: 70 years of the Queen in 70 ...,platinum jubilee: 70 years of the queen in 70 ...


In [6]:
# remove stop words from titles
en_stopwords = stopwords.words('english')
titles['no_stopwords_title'] = titles['lowercase'].apply\
                                (lambda x: ' '.join([word for word in x.split() if word not in en_stopwords]))
titles.head()

,title,lowercase,no_stopwords_title
0,Can I refuse to work?,can i refuse to work?,refuse work?
1,'Liz Truss the Brief?' World reacts to UK poli...,'liz truss the brief?' world reacts to uk poli...,'liz truss brief?' world reacts uk political t...
2,Rationing energy is nothing new for off-grid c...,rationing energy is nothing new for off-grid c...,rationing energy nothing new off-grid community
3,The hunt for superyachts of sanctioned Russian...,the hunt for superyachts of sanctioned russian...,hunt superyachts sanctioned russian oligarchs
4,Platinum Jubilee: 70 years of the Queen in 70 ...,platinum jubilee: 70 years of the queen in 70 ...,platinum jubilee: 70 years queen 70 seconds


In [7]:
# remove punctuations
titles['no_punct'] = titles.apply(lambda x: re.sub(r"([^\w\s])","",x['no_stopwords_title']),axis=1)
titles.head()

,title,lowercase,no_stopwords_title,no_punct
0,Can I refuse to work?,can i refuse to work?,refuse work?,refuse work
1,'Liz Truss the Brief?' World reacts to UK poli...,'liz truss the brief?' world reacts to uk poli...,'liz truss brief?' world reacts uk political t...,liz truss brief world reacts uk political turmoil
2,Rationing energy is nothing new for off-grid c...,rationing energy is nothing new for off-grid c...,rationing energy nothing new off-grid community,rationing energy nothing new offgrid community
3,The hunt for superyachts of sanctioned Russian...,the hunt for superyachts of sanctioned russian...,hunt superyachts sanctioned russian oligarchs,hunt superyachts sanctioned russian oligarchs
4,Platinum Jubilee: 70 years of the Queen in 70 ...,platinum jubilee: 70 years of the queen in 70 ...,platinum jubilee: 70 years queen 70 seconds,platinum jubilee 70 years queen 70 seconds


In [8]:
titles['tokens_row'] = titles.apply(lambda x: word_tokenize(x['title']), axis=1)
titles['clean_tokens'] = titles.apply(lambda x: word_tokenize(x['no_stopwords_title']), axis=1)
#titles.head()

In [11]:
# Lemmatizing
lemmatizer = WordNetLemmatizer()
titles['clean_tokens_lemmatizer'] = titles['clean_tokens'].apply(lambda x: [lemmatizer.lemmatize(token) for token in x])
titles.head()

,title,lowercase,no_stopwords_title,no_punct,tokens_row,clean_tokens,clean_tokens_lemmatizer
0,Can I refuse to work?,can i refuse to work?,refuse work?,refuse work,"[Can, I, refuse, to, work, ?]","[refuse, work, ?]","[refuse, work, ?]"
1,'Liz Truss the Brief?' World reacts to UK poli...,'liz truss the brief?' world reacts to uk poli...,'liz truss brief?' world reacts uk political t...,liz truss brief world reacts uk political turmoil,"['Liz, Truss, the, Brief, ?, ', World, reacts,...","['liz, truss, brief, ?, ', world, reacts, uk, ...","['liz, truss, brief, ?, ', world, reacts, uk, ..."
2,Rationing energy is nothing new for off-grid c...,rationing energy is nothing new for off-grid c...,rationing energy nothing new off-grid community,rationing energy nothing new offgrid community,"[Rationing, energy, is, nothing, new, for, off...","[rationing, energy, nothing, new, off-grid, co...","[rationing, energy, nothing, new, off-grid, co..."
3,The hunt for superyachts of sanctioned Russian...,the hunt for superyachts of sanctioned russian...,hunt superyachts sanctioned russian oligarchs,hunt superyachts sanctioned russian oligarchs,"[The, hunt, for, superyachts, of, sanctioned, ...","[hunt, superyachts, sanctioned, russian, oliga...","[hunt, superyachts, sanctioned, russian, oliga..."
4,Platinum Jubilee: 70 years of the Queen in 70 ...,platinum jubilee: 70 years of the queen in 70 ...,platinum jubilee: 70 years queen 70 seconds,platinum jubilee 70 years queen 70 seconds,"[Platinum, Jubilee, :, 70, years, of, the, Que...","[platinum, jubilee, :, 70, years, queen, 70, s...","[platinum, jubilee, :, 70, year, queen, 70, se..."


In [17]:
tokens_raw_list = sum(titles['tokens_row'], [])
tokens_clean_list = sum(titles['clean_tokens_lemmatizer'], [])

### Part Of Speech Tagging (POST)

In [19]:
nlp = spacy.load("en_core_web_sm")
spacy_doc = nlp(' '.join(tokens_raw_list))

In [20]:
pos_df = pd.DataFrame(columns=['token', 'pos_tag'])
for token in spacy_doc:
    pos_df = pd.concat([pos_df, pd.DataFrame.from_records([{'token': token.text, 'pos_tag': token.pos_}])], ignore_index=True)

In [22]:
# token frequency counts
pos_df_counts = pos_df.groupby(['token','pos_tag']).size().reset_index(name='counts').sort_values(by='counts',ascending=False) 
pos_df_counts.head()

,token,pos_tag,counts
95,:,PUNCT,543
8,',PUNCT,300
2897,in,ADP,187
4082,to,PART,175
3268,of,ADP,172


In [29]:
nouns = pos_df_counts[pos_df_counts.pos_tag == 'NOUN']
nouns.head()

,token,pos_tag,counts
4267,war,NOUN,35
3552,record,NOUN,15
3416,police,NOUN,14
4316,win,NOUN,14
4356,year,NOUN,14


In [30]:
verbs = pos_df_counts[pos_df_counts.pos_tag == 'VERB']
verbs.head()

,token,pos_tag,counts
3687,says,VERB,30
9,',VERB,14
2670,found,VERB,13
4317,win,VERB,12
4324,wins,VERB,10


### Named Entity Recognition

In [35]:
#Create a NER dataframe and process the entities with the spacy_doc
ner_df = pd.DataFrame(columns=['token', 'ner_tag'])
#ner_df.head()

for token in spacy_doc.ents:
    if pd.isna(token.label_) is False:
        ner_df = pd.concat([ner_df, pd.DataFrame.from_records([{'token': token.text, 'ner_tag': token.label_}])],
                          ignore_index=True)
ner_df.head()

,token,ner_tag
0,Liz Truss,PERSON
1,UK,GPE
2,Rationing,PRODUCT
3,superyachts,CARDINAL
4,Russian,NORP


In [37]:
ner_df_counts = ner_df.groupby(['token', 'ner_tag']).size().reset_index(name='counts').sort_values(by='counts',ascending=False)
ner_df_counts.head()

,token,ner_tag,counts
965,Ukraine,GPE,47
955,UK,GPE,36
329,England,GPE,32
819,Russian,NORP,20
957,US,GPE,19
